2026-04-08

Shahlo Solieva

Notebook to calculate distances between ALL HEAVY ATOMS on antibody to ALL HEAVY ATOMS of antigen glycans and protein. Excluding hydrogen atoms.

In [6]:
import numpy as np
import mdtraj as md
import matplotlib.pyplot as plt
import itertools
import pandas as pd

BASE_PATH='/mnt/pure/bowmanlab/WG-shahlo/projects/Ab_dynamics/analysis/distance_to_antigen/'

# Helper Functions

In [7]:
def get_resis(sys_path, start_of_VH, end_of_VH, start_of_VL, end_of_VL):
    # define inputs directory path and print out all input variables.
    cluster_dir_path = f"/mnt/pure/bowmanlab/WG-shahlo/projects/Ab_dynamics/analysis/clustering/radius_15_variable/data/{sys_path}/"
    print("\tINPUTS:", sys_path,start_of_VH, end_of_VH, start_of_VL, end_of_VL)

    # select alpha carbons from the variable region:
    #pdb_path = f'{cluster_dir_path}prot_masses_uniform.pdb'
    pdb_path = f'{cluster_dir_path}prot_masses.pdb'
    pdb = md.load(pdb_path)
    all_ca_atoms = pdb.topology.select(f'chainid 0 and residue {start_of_VH} to {end_of_VH} and name CA or chainid 1 and residue {start_of_VL} to {end_of_VL} and name CA')

    # check to make sure that the atoms / atom indicies selected are from the correct residues.
    variable_resi_names = []
    for i in range(len(all_ca_atoms)):
        variable_resi_names.append(pdb.topology.atom(all_ca_atoms[i]).residue)
    variable_HC_LC_resi_names = [variable_resi_names[:end_of_VH], variable_resi_names[end_of_VH:]]
    return variable_HC_LC_resi_names

In [8]:
colors=['#80bfff','#004080']

def import_data_apo(system_name):
    base_path = '/mnt/pure/bowmanlab/WG-shahlo/projects/Ab_dynamics/analysis/per_resi_RMSD'
    rmsd_data = np.load(f'{base_path}/data/apo/{system_name}/VHC_VLC_CA_RMSD_per_resi_to_average_structure.npy', allow_pickle=True)
    return rmsd_data 

def get_HC_rmsd(rmsd_data, vh_end):
    return rmsd_data[:vh_end]*10
def get_LC_rmsd(rmsd_data, vh_end):
    return rmsd_data[vh_end:]*10

def calculate_min_dist_values(pdb_path, HC, LC, antigen_, vh_end, vl_end):
    '''
    examples: 
        antigen = '8 10'
        HC = '5' 
        LC = '1' 
    '''
    pdb = md.load(pdb_path)
    heavy_chain_CA = pdb.topology.select(f'chainid {HC} and name CA')
    light_chain_CA = pdb.topology.select(f'chainid {LC} and name CA')
    antigen = pdb.topology.select(f'resname BMA MAN NAG and not element H')
    print(len(heavy_chain_CA), len(light_chain_CA), len(antigen))
    HC_glycan_minDists = []
    HC_resis = []
    for i in range(len(heavy_chain_CA)):
        Ab_residue_n = int(str(pdb.topology.atom(heavy_chain_CA[i]).residue)[3:]) 
        HC_resis.append(pdb.topology.atom(heavy_chain_CA[i]).residue)
        Ab_residue_atoms = pdb.topology.select(f'chainid {HC} and residue {Ab_residue_n} and not element H')
        HC_glycan_pairs = list(itertools.product(Ab_residue_atoms, antigen))
        HC_glycan_minDists.append(np.min(md.compute_distances(pdb, HC_glycan_pairs)[0]))
    LC_glycan_minDists = []
    LC_resis = []
    for i in range(len(light_chain_CA)):
        Ab_residue_n = int(str(pdb.topology.atom(light_chain_CA[i]).residue)[3:]) 
        LC_resis.append(pdb.topology.atom(light_chain_CA[i]).residue)
        Ab_residue_atoms = pdb.topology.select(f'chainid {LC} and residue {Ab_residue_n} and not element H')
        LC_glycan_pairs = list(itertools.product(Ab_residue_atoms, antigen))
        LC_glycan_minDists.append(np.min(md.compute_distances(pdb, LC_glycan_pairs)[0]))
    HC_glycan_minDists = np.array(HC_glycan_minDists[:vh_end])*10 # nm to Angstrom, and only save VHC values. 
    LC_glycan_minDists = np.array(LC_glycan_minDists[:vl_end])*10 # nm to Angstrom, and only save VLC values.
    return HC_glycan_minDists, LC_glycan_minDists, HC_resis[:vh_end], LC_resis[:vl_end]

In [9]:
def combined_steps_to_import_data(system_name, HC_start, HC_end, LC_start, LC_end):
    '''
    examples:
        system_name='CAP256_mature_VC'
        HC_start, HC_end, LC_start, LC_end = 1, 145, 1, 111 # Mature CAP256: "1" "145" "1" "111"
    '''
    print(f'Importing RMSD data for {system_name}')
    mature_VHC_VLC_resi_names = get_resis(f"apo/{system_name}",  HC_start, HC_end, LC_start, LC_end)
    rmsd_data = import_data_apo(system_name)
    mature_HC_rmsd_values, mature_LC_rmsd_values = get_HC_rmsd(rmsd_data, HC_end), get_LC_rmsd(rmsd_data, HC_end)  
    return  mature_VHC_VLC_resi_names, mature_HC_rmsd_values, mature_LC_rmsd_values


In [10]:
def check_pdb_lengths_mature_Abs(mature_VHC_VLC_resi_names, pdb_HC_resis, pdb_LC_resis, n=2):
    '''
    Check PDB lengths to see if any residues are missing.
    '''
    print('\t\tVariable Heavy Chain:\tVariable Light Chain:')
    print('Mature Length:\t', len(mature_VHC_VLC_resi_names[0]), '\t\t\t',len(mature_VHC_VLC_resi_names[1]))
    print('   PDB Length:\t', len(pdb_HC_resis), '\t\t\t',len(pdb_LC_resis))
    print(' Mature Start:\t',mature_VHC_VLC_resi_names[0][0:n], '\t\t',mature_VHC_VLC_resi_names[1][0:n])
    print('    PDB Start:\t',pdb_HC_resis[0:n],'\t\t',pdb_LC_resis[0:n])
    print('   Mature End:\t',mature_VHC_VLC_resi_names[0][-n:], '\t',mature_VHC_VLC_resi_names[1][-n:])
    print('      PDB End:\t',pdb_HC_resis[-n:],'\t',pdb_LC_resis[-n:],'\n')

# CAP256

## import and calculate data. 

In [11]:
# -------------------------------------------------------
# Import RMSD data for the mature antibody. (Mature antibody first because the PDB we are comparing to is the mature antibody)
system_name='CAP256_mature_VC'
HC_start, HC_end, LC_start, LC_end = 1, 145, 1, 111 # Mature CAP256: "1" "145" "1" "111"
mature_VHC_VLC_resi_names, mature_HC_rmsd_values, mature_LC_rmsd_values = combined_steps_to_import_data(system_name, HC_start, HC_end, LC_start, LC_end)

# Calculate minimum distances based on the published PDB structure:
print('Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.')
pdb_path = f'{BASE_PATH}structures/6vtt_cap256_clean_renumbered.pdb'
HC, LC, antigen = '3', '4', '0 1 2'
pdb_HC_glycan_minDists, pdb_LC_glycan_minDists, pdb_HC_resis, pdb_LC_resis = calculate_min_dist_values(pdb_path, HC, LC, antigen, HC_end, LC_end)
# -------------------------------------------------------
# Align the PDB data to the mature antibody -- some PDB structures are missing the first few residues or last few resis. 
check_pdb_lengths_mature_Abs(mature_VHC_VLC_resi_names, pdb_HC_resis, pdb_LC_resis, n=3)

# Need to add one resi at the start and at the end for the HC.
pdb_HC_resis_fixed = np.insert(pdb_HC_resis, [0,len(pdb_HC_resis)], ['GLN1','SER145'])
pdb_HC_glycan_minDists_fixed = np.insert(pdb_HC_glycan_minDists, [0, len(pdb_HC_glycan_minDists)], [None,None]) 
for i in range(HC_end):
    if str(pdb_HC_resis_fixed[i]) != str(mature_VHC_VLC_resi_names[0][i]):
        print('ERROR: HC misalignment',i, pdb_HC_resis_fixed[i], mature_VHC_VLC_resi_names[0][i])

# Need to add 3 resis at the end for the LC.
pdb_LC_resis_fixed = np.insert(pdb_LC_resis, [len(pdb_LC_resis), len(pdb_LC_resis), len(pdb_LC_resis)], ['ILE109', 'VAL110', 'LEU111'])
pdb_LC_glycan_minDists_fixed = np.insert(pdb_LC_glycan_minDists, [len(pdb_LC_glycan_minDists), len(pdb_LC_glycan_minDists), len(pdb_LC_glycan_minDists)], [None,None,None]) # to account for the added 'gln1' in HC resis. 
for i in range(LC_end):
    if str(pdb_LC_resis_fixed[i]) != str(mature_VHC_VLC_resi_names[1][i]):
        print('ERROR: LC misalignment',i, pdb_LC_resis_fixed[i], mature_VHC_VLC_resi_names[1][i])
# -------------------------------------------------------
# Import RMSD data for the naive antibody now.
system_name='CAP256_naive_VC'
HC_start, HC_end, LC_start, LC_end = 1, 144, 1, 111 # Naive CAP256
naive_VHC_VLC_resi_names, naive_HC_rmsd_values, naive_LC_rmsd_values = combined_steps_to_import_data(system_name, HC_start, HC_end, LC_start, LC_end)

Importing RMSD data for CAP256_mature_VC
	INPUTS: apo/CAP256_mature_VC 1 145 1 111
Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.


/home/ssolieva/miniconda/envs/pyemma/lib/python3.11/site-packages/mdtraj/formats/pdb/pdbfile.py:200: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn('Unlikely unit cell vectors detected in PDB file likely '


143 108 295
		Variable Heavy Chain:	Variable Light Chain:
Mature Length:	 145 			 111
   PDB Length:	 143 			 108
 Mature Start:	 [GLN1, VAL2, GLN3] 		 [GLN1, SER2, VAL3]
    PDB Start:	 [VAL2, GLN3, LEU4] 		 [GLN1, SER2, VAL3]
   Mature End:	 [VAL143, SER144, SER145] 	 [ILE109, VAL110, LEU111]
      PDB End:	 [THR142, VAL143, SER144] 	 [THR106, LYS107, VAL108] 

ERROR: HC misalignment 111 TYS112 TYR112
ERROR: HC misalignment 112 TYS113 TYR113
Importing RMSD data for CAP256_naive_VC
	INPUTS: apo/CAP256_naive_VC 1 144 1 111


## Align Naive and Mature/PDB (sequence-based alignment)

In [12]:
# Light chain for CAP256 is already aligned. Just save the output. 
# Heavy chain for CAP256 needs to be aligned. Adding to naive and mature. Naive needs 1 extra None. 
# add 'None' as gaps for the sequence alignment
naive_HC_rmsd_values_fixed = np.insert(naive_HC_rmsd_values,       [118,118,120,125,129], None)
naive_VHC_resi_names_fixed = np.insert(naive_VHC_VLC_resi_names[0],[118,118,120,125,129], None)
print(len(naive_HC_rmsd_values_fixed), len(naive_VHC_resi_names_fixed))

mature_HC_rmsd_values_fixed=np.insert(mature_HC_rmsd_values,       [115,115,124,132], None)
mature_VHC_resi_names_fixed=np.insert(mature_VHC_VLC_resi_names[0],[115,115,124,132], None)
print(len(mature_HC_rmsd_values_fixed), len(mature_VHC_resi_names_fixed))

# PDB data alignment:
pdb_HC_resis_fixed_fixed           =np.insert(pdb_HC_resis_fixed,          [115,115,124,132], None)
pdb_HC_glycan_minDists_fixed_fixed=np.insert(pdb_HC_glycan_minDists_fixed, [115,115,124,132], None)
print(len(pdb_HC_resis_fixed_fixed), len(pdb_HC_glycan_minDists_fixed_fixed))

pdb_LC_resis_fixed = np.insert(pdb_LC_resis, [108,108,108], None)
pdb_LC_glycan_minDists_fixed  = np.insert(pdb_LC_glycan_minDists, [108,108,108], None)
print(len(pdb_LC_resis_fixed), len(pdb_LC_glycan_minDists_fixed))

149 149
149 149
149 149
111 111


## Save output for plotting later. 

In [13]:
data={"pdb_mature_HC_resi":pdb_HC_resis_fixed_fixed,
      "pdb_mature_HC_glycan_minDists":pdb_HC_glycan_minDists_fixed_fixed,
      "naive_HC_resi": naive_VHC_resi_names_fixed,
      "naive_HC_rmsd": naive_HC_rmsd_values_fixed,
      "mature_HC_resi": mature_VHC_resi_names_fixed,
      "mature_HC_rmsd": mature_HC_rmsd_values_fixed}
df = pd.DataFrame(data)
output_filename = f"{BASE_PATH}data/to_glycan/CAP256_HC.csv"
df.to_csv(output_filename, index=False)

data={"pdb_mature_LC_resi":pdb_LC_resis_fixed,
      "pdb_mature_LC_glycan_minDists":pdb_LC_glycan_minDists_fixed,
      "naive_LC_resi": naive_VHC_VLC_resi_names[1],
      "naive_LC_rmsd": naive_LC_rmsd_values,
      "mature_LC_resi": mature_VHC_VLC_resi_names[1],
      "mature_LC_rmsd": mature_LC_rmsd_values}
df = pd.DataFrame(data)
output_filename = f"{BASE_PATH}data/to_glycan/CAP256_LC.csv"
df.to_csv(output_filename, index=False)

# 1-18

## import and calculate data. 

In [14]:
# -------------------------------------------------------
# Import RMSD data for the mature antibody. (Mature antibody first because the PDB we are comparing to is the mature antibody)
system_name='Ab118_mature_VC'
HC_start, HC_end, LC_start, LC_end = 1, 131, 1, 108
mature_VHC_VLC_resi_names, mature_HC_rmsd_values, mature_LC_rmsd_values = combined_steps_to_import_data(system_name, HC_start, HC_end, LC_start, LC_end)

# Calculate minimum distances based on the published PDB structure:
print('Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.')
pdb_path = f'{BASE_PATH}structures/6udj_1-18_clean_renumbered.pdb'
HC, LC, antigen = '5', '1', '8 10'
pdb_HC_glycan_minDists, pdb_LC_glycan_minDists, pdb_HC_resis, pdb_LC_resis = calculate_min_dist_values(pdb_path, HC, LC, antigen, HC_end, LC_end)
# -------------------------------------------------------
# Align the PDB data to the mature antibody -- some PDB structures are missing the first few residues or last few resis. 
check_pdb_lengths_mature_Abs(mature_VHC_VLC_resi_names, pdb_HC_resis, pdb_LC_resis, n=2)
# Need to add one resi at the start of the HC.
pdb_HC_resis_fixed = np.insert(pdb_HC_resis, [0], ['GLN1'])
pdb_HC_glycan_minDists_fixed = np.insert(pdb_HC_glycan_minDists, [0], [None]) 
for i in range(HC_end):
    if str(pdb_HC_resis_fixed[i]) != str(mature_VHC_VLC_resi_names[0][i]):
        print('ERROR: HC misalignment',i, pdb_HC_resis_fixed[i], mature_VHC_VLC_resi_names[0][i])
# LC pdb is already aligned to mature. 
# -------------------------------------------------------
# Import RMSD data for the naive antibody now.
system_name='Ab118_naive_VC'
HC_start, HC_end, LC_start, LC_end = 1, 125, 1, 108
naive_VHC_VLC_resi_names, naive_HC_rmsd_values, naive_LC_rmsd_values = combined_steps_to_import_data(system_name, HC_start, HC_end, LC_start, LC_end)
# -------------------------------------------------------

Importing RMSD data for Ab118_mature_VC
	INPUTS: apo/Ab118_mature_VC 1 131 1 108
Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.
130 108 628
		Variable Heavy Chain:	Variable Light Chain:
Mature Length:	 131 			 108
   PDB Length:	 130 			 108
 Mature Start:	 [GLN1, GLY2] 		 [GLU1, VAL2]
    PDB Start:	 [GLY2, ARG3] 		 [GLU1, VAL2]
   Mature End:	 [SER130, SER131] 	 [LYS107, LYS108]
      PDB End:	 [SER130, SER131] 	 [LYS107, LYS108] 

Importing RMSD data for Ab118_naive_VC
	INPUTS: apo/Ab118_naive_VC 1 125 1 108


## Align Naive and Mature/PDB (sequence-based alignment)

In [15]:
# Light chain for 1-18 is already aligned. Just save the output. 
# Heavy chain for 1-18 needs to be aligned. Adding a total of 6 'None' values to naive sequence. 
print(len(naive_HC_rmsd_values), len(mature_HC_rmsd_values))
# add 'None' as gaps for the sequence alignment
naive_HC_rmsd_values_fixed = np.insert(naive_HC_rmsd_values,       [26,26,27,27,27,27], None)
naive_VHC_resi_names_fixed = np.insert(naive_VHC_VLC_resi_names[0],[26,26,27,27,27,27], None)
print(len(naive_HC_rmsd_values_fixed), len(mature_HC_rmsd_values))

125 131
131 131


## Save output for plotting later. 

In [16]:
data={"pdb_mature_HC_resi":pdb_HC_resis_fixed,
      "pdb_mature_HC_glycan_minDists":pdb_HC_glycan_minDists_fixed,
      "naive_HC_resi": naive_VHC_resi_names_fixed,
      "naive_HC_rmsd": naive_HC_rmsd_values_fixed,
      "mature_HC_resi": mature_VHC_VLC_resi_names[0],
      "mature_HC_rmsd": mature_HC_rmsd_values}
df = pd.DataFrame(data)
output_filename = f"{BASE_PATH}data/to_glycan/Ab118_HC.csv"
df.to_csv(output_filename, index=False)

data = {"pdb_mature_LC_resi":pdb_LC_resis,
        "pdb_mature_LC_glycan_minDists":pdb_LC_glycan_minDists,
        "naive_LC_resi": naive_VHC_VLC_resi_names[1],
        "naive_LC_rmsd": naive_LC_rmsd_values,
        "mature_LC_resi": mature_VHC_VLC_resi_names[1],
        "mature_LC_rmsd": mature_LC_rmsd_values}
df = pd.DataFrame(data)
output_filename = f"{BASE_PATH}data/to_glycan/Ab118_LC.csv"
df.to_csv(output_filename, index=False)

# CH103

## Import and Calculate data

In [17]:
# -------------------------------------------------------
# Import RMSD data for the mature antibody. (Mature antibody first because the PDB we are comparing to is the mature antibody)
system_name='CH103_mature_VC'
HC_start, HC_end, LC_start, LC_end = 1, 121, 1, 105 # Mature 
mature_VHC_VLC_resi_names, mature_HC_rmsd_values, mature_LC_rmsd_values = combined_steps_to_import_data(system_name, HC_start, HC_end, LC_start, LC_end)

# Calculate minimum distances based on the published PDB structure:
print('Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.')
pdb_path = f'{BASE_PATH}structures/9d7o_CH103_clean_renumbered.pdb'
HC, LC, antigen = '8', '6', '2 3'
pdb_HC_glycan_minDists, pdb_LC_glycan_minDists, pdb_HC_resis, pdb_LC_resis = calculate_min_dist_values(pdb_path, HC, LC, antigen, HC_end, LC_end)
# -------------------------------------------------------
# Align the PDB data to the mature antibody -- some PDB structures are missing the first few residues or last few resis. 
check_pdb_lengths_mature_Abs(mature_VHC_VLC_resi_names, pdb_HC_resis, pdb_LC_resis, n=2)
# HC pdb is already aligned to mature. 
# Need to add 1 resis at the start and remove one at end for the LC.
pdb_LC_resis_fixed = np.insert(pdb_LC_resis[:-1], [0], ['SER1'])
pdb_LC_glycan_minDists_fixed = np.insert(pdb_LC_glycan_minDists[:-1], 0, None) 
for i in range(LC_end):
    if str(pdb_LC_resis_fixed[i]) != str(mature_VHC_VLC_resi_names[1][i]):
        print('ERROR: LC misalignment',i, pdb_LC_resis_fixed[i], mature_VHC_VLC_resi_names[1][i])
# -------------------------------------------------------
# Import RMSD data for the naive antibody now.
system_name='CH103_naive_VC'
HC_start, HC_end, LC_start, LC_end = 1, 121, 1, 108 # Naive 
naive_VHC_VLC_resi_names, naive_HC_rmsd_values, naive_LC_rmsd_values = combined_steps_to_import_data(system_name, HC_start, HC_end, LC_start, LC_end)
# -------------------------------------------------------

Importing RMSD data for CH103_mature_VC
	INPUTS: apo/CH103_mature_VC 1 121 1 105
Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.
176 167 492
		Variable Heavy Chain:	Variable Light Chain:
Mature Length:	 121 			 105
   PDB Length:	 121 			 105
 Mature Start:	 [GLN1, VAL2] 		 [SER1, TYR2]
    PDB Start:	 [GLN1, VAL2] 		 [TYR2, GLU3]
   Mature End:	 [THR120, ALA121] 	 [LEU104, ARG105]
      PDB End:	 [THR120, ALA121] 	 [GLY105, GLN106] 

ERROR: LC misalignment 104 GLY105 ARG105
Importing RMSD data for CH103_naive_VC
	INPUTS: apo/CH103_naive_VC 1 121 1 108


## Align Naive and Mature/PDB (sequence-based alignment)

In [18]:
# Heavy chain for CH103 is already aligned. Just save the output. 
# Light chain for CH103 needs to be aligned. Adding a total of 3 'None' values to mature sequence. Also to PDB mature sequence. 
mature_LC_rmsd_values_fixed=np.insert(mature_LC_rmsd_values,       [27,28,28], None)
mature_VLC_resi_names_fixed=np.insert(mature_VHC_VLC_resi_names[1],[27,28,28], None)

pdb_LC_resis_fixed_fixed          =np.insert(pdb_LC_resis_fixed,          [27,28,28], None)
pdb_LC_glycan_minDists_fixed_fixed=np.insert(pdb_LC_glycan_minDists_fixed,[27,28,28], None)

## Save output for plotting later. 

In [19]:
data={"pdb_mature_HC_resi":pdb_HC_resis,
      "pdb_mature_HC_glycan_minDists":pdb_HC_glycan_minDists,
      "naive_HC_resi": naive_VHC_VLC_resi_names[0],
      "naive_HC_rmsd": naive_HC_rmsd_values,
      "mature_HC_resi": mature_VHC_VLC_resi_names[0],
      "mature_HC_rmsd": mature_HC_rmsd_values}
df = pd.DataFrame(data)
output_filename = f"{BASE_PATH}data/to_glycan/CH103_HC.csv"
df.to_csv(output_filename, index=False)

data={"pdb_mature_LC_resi":pdb_LC_resis_fixed_fixed,
      "pdb_mature_LC_glycan_minDists":pdb_LC_glycan_minDists_fixed_fixed,
      "naive_LC_resi": naive_VHC_VLC_resi_names[1],
      "naive_LC_rmsd": naive_LC_rmsd_values,
      "mature_LC_resi": mature_VLC_resi_names_fixed,
      "mature_LC_rmsd": mature_LC_rmsd_values_fixed}
df = pd.DataFrame(data)
output_filename = f"{BASE_PATH}data/to_glycan/CH103_LC.csv"
df.to_csv(output_filename, index=False)

# EPTC112

## Import and Calculate data

In [20]:
lineage = 'EPTC112'
# -------------------------------------------------------
# Import RMSD data for the mature antibody. (Mature antibody first because the PDB we are comparing to is the mature antibody)
system_name=f'{lineage}_mature_VC'
HC_start, HC_end, LC_start, LC_end = 1, 121, 1, 110 
mature_VHC_VLC_resi_names, mature_HC_rmsd_values, mature_LC_rmsd_values = combined_steps_to_import_data(system_name, HC_start, HC_end, LC_start, LC_end)

# Calculate minimum distances based on the published PDB structure:
print('Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.')
pdb_path = f'{BASE_PATH}structures/8c8t_{lineage}_clean_renumbered.pdb'
HC, LC, antigen = '1', '3', '0'
pdb_HC_glycan_minDists, pdb_LC_glycan_minDists, pdb_HC_resis, pdb_LC_resis = calculate_min_dist_values(pdb_path, HC, LC, antigen, HC_end, LC_end)
# -------------------------------------------------------
# Align the PDB data to the mature antibody -- some PDB structures are missing the first few residues or last few resis. 
check_pdb_lengths_mature_Abs(mature_VHC_VLC_resi_names, pdb_HC_resis, pdb_LC_resis, n=3)
# Need to add one resi at the start and at the end for the HC.
pdb_HC_resis_fixed           = np.insert(pdb_HC_resis,           [0, len(pdb_HC_resis)],          ['GLN1','SER121'])
pdb_HC_glycan_minDists_fixed = np.insert(pdb_HC_glycan_minDists, [0, len(pdb_HC_glycan_minDists)], None) 
for i in range(HC_end):
    if str(pdb_HC_resis_fixed[i]) != str(mature_VHC_VLC_resi_names[0][i]):
        print('ERROR: HC misalignment',i, pdb_HC_resis_fixed[i], mature_VHC_VLC_resi_names[0][i])
# Need to add 2 resis for the LC.
pdb_LC_resis_fixed           = np.insert(pdb_LC_resis,           [0,len(pdb_LC_resis)],          ['GLN1','LEU110'])
pdb_LC_glycan_minDists_fixed = np.insert(pdb_LC_glycan_minDists, [0,len(pdb_LC_glycan_minDists)], None)
for i in range(LC_end):
    if str(pdb_LC_resis_fixed[i]) != str(mature_VHC_VLC_resi_names[1][i]):
        print('ERROR: LC misalignment',i, pdb_LC_resis_fixed[i], mature_VHC_VLC_resi_names[1][i])
# -------------------------------------------------------
# Import RMSD data for the naive antibody now.
system_name=f'{lineage}_naive_VC'
HC_start, HC_end, LC_start, LC_end = 1, 121, 1, 110 
naive_VHC_VLC_resi_names, naive_HC_rmsd_values, naive_LC_rmsd_values = combined_steps_to_import_data(system_name, HC_start, HC_end, LC_start, LC_end)

Importing RMSD data for EPTC112_mature_VC
	INPUTS: apo/EPTC112_mature_VC 1 121 1 110
Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.
119 108 201
		Variable Heavy Chain:	Variable Light Chain:
Mature Length:	 121 			 110
   PDB Length:	 119 			 108
 Mature Start:	 [GLN1, VAL2, GLN3] 		 [GLN1, SER2, VAL3]
    PDB Start:	 [VAL2, GLN3, LEU4] 		 [SER2, VAL3, LEU4]
   Mature End:	 [VAL119, SER120, SER121] 	 [THR108, VAL109, LEU110]
      PDB End:	 [VAL118, VAL119, SER120] 	 [VAL107, THR108, VAL109] 

Importing RMSD data for EPTC112_naive_VC
	INPUTS: apo/EPTC112_naive_VC 1 121 1 110


## Save output for plotting later. 

In [21]:
data={"pdb_mature_HC_resi":pdb_HC_resis_fixed,
      "pdb_mature_HC_glycan_minDists":pdb_HC_glycan_minDists_fixed,
      "naive_HC_resi": naive_VHC_VLC_resi_names[0],
      "naive_HC_rmsd": naive_HC_rmsd_values,
      "mature_HC_resi": mature_VHC_VLC_resi_names[0],
      "mature_HC_rmsd": mature_HC_rmsd_values}
df = pd.DataFrame(data)
output_filename = f"{BASE_PATH}data/to_glycan/{lineage}_HC.csv"
df.to_csv(output_filename, index=False)

data={"pdb_mature_LC_resi":pdb_LC_resis_fixed,
      "pdb_mature_LC_glycan_minDists":pdb_LC_glycan_minDists_fixed,
      "naive_LC_resi": naive_VHC_VLC_resi_names[1],
      "naive_LC_rmsd": naive_LC_rmsd_values,
      "mature_LC_resi": mature_VHC_VLC_resi_names[1],
      "mature_LC_rmsd": mature_LC_rmsd_values}
df = pd.DataFrame(data)
output_filename = f"{BASE_PATH}data/to_glycan/{lineage}_LC.csv"
df.to_csv(output_filename, index=False)

# CH65

## Import and Calculate data

In [22]:
lineage = 'CH65'
# -------------------------------------------------------
# Import RMSD data for the mature antibody. (Mature antibody first because the PDB we are comparing to is the mature antibody)
system_name=f'{lineage}_mature_VC'
HC_start, HC_end, LC_start, LC_end = 1, 126, 1, 108
mature_VHC_VLC_resi_names, mature_HC_rmsd_values, mature_LC_rmsd_values = combined_steps_to_import_data(system_name, HC_start, HC_end, LC_start, LC_end)

# Calculate minimum distances based on the published PDB structure:
print('Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.')
pdb_path = f'{BASE_PATH}structures/5ugy_{lineage}_clean_renumbered.pdb'
HC, LC, antigen = '1', '2', '0'
pdb_HC_glycan_minDists, pdb_LC_glycan_minDists, pdb_HC_resis, pdb_LC_resis = calculate_min_dist_values(pdb_path, HC, LC, antigen, HC_end, LC_end)
# -------------------------------------------------------
# Align the PDB data to the mature antibody -- some PDB structures are missing the first few residues or last few resis. 
check_pdb_lengths_mature_Abs(mature_VHC_VLC_resi_names, pdb_HC_resis, pdb_LC_resis, n=3)
# HC PDB is already aligned.
# Need to add 1 and remove 1 resi for the LC.
pdb_LC_resis_fixed           = np.insert(pdb_LC_resis[:-1],           0,'GLN1')
pdb_LC_glycan_minDists_fixed = np.insert(pdb_LC_glycan_minDists[:-1], 0, None)
for i in range(LC_end):
    if str(pdb_LC_resis_fixed[i]) != str(mature_VHC_VLC_resi_names[1][i]):
        print('ERROR: LC misalignment',i, pdb_LC_resis_fixed[i], mature_VHC_VLC_resi_names[1][i])
# -------------------------------------------------------
# Import RMSD data for the naive antibody now.
system_name=f'{lineage}_naive_VC'
HC_start, HC_end, LC_start, LC_end = 1, 126, 1, 108
naive_VHC_VLC_resi_names, naive_HC_rmsd_values, naive_LC_rmsd_values = combined_steps_to_import_data(system_name, HC_start, HC_end, LC_start, LC_end)

Importing RMSD data for CH65_mature_VC
	INPUTS: apo/CH65_mature_VC 1 126 1 108
Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.
221 210 123
		Variable Heavy Chain:	Variable Light Chain:
Mature Length:	 126 			 108
   PDB Length:	 126 			 108
 Mature Start:	 [GLU1, VAL2, GLN3] 		 [GLN1, SER2, VAL3]
    PDB Start:	 [GLU1, VAL2, GLN3] 		 [SER2, VAL3, LEU4]
   Mature End:	 [VAL124, SER125, SER126] 	 [THR106, VAL107, LEU108]
      PDB End:	 [VAL124, SER125, SER126] 	 [VAL107, LEU108, GLY109] 

Importing RMSD data for CH65_naive_VC
	INPUTS: apo/CH65_naive_VC 1 126 1 108


## Save output for plotting later. 

In [23]:
data={"pdb_mature_HC_resi":pdb_HC_resis,
      "pdb_mature_HC_glycan_minDists":pdb_HC_glycan_minDists,
      "naive_HC_resi": naive_VHC_VLC_resi_names[0],
      "naive_HC_rmsd": naive_HC_rmsd_values,
      "mature_HC_resi": mature_VHC_VLC_resi_names[0],
      "mature_HC_rmsd": mature_HC_rmsd_values}
df = pd.DataFrame(data)
output_filename = f"{BASE_PATH}data/to_glycan/{lineage}_HC.csv"
df.to_csv(output_filename, index=False)

data={"pdb_mature_LC_resi":pdb_LC_resis_fixed,
      "pdb_mature_LC_glycan_minDists":pdb_LC_glycan_minDists_fixed,
      "naive_LC_resi": naive_VHC_VLC_resi_names[1],
      "naive_LC_rmsd": naive_LC_rmsd_values,
      "mature_LC_resi": mature_VHC_VLC_resi_names[1],
      "mature_LC_rmsd": mature_LC_rmsd_values}
df = pd.DataFrame(data)
output_filename = f"{BASE_PATH}data/to_glycan/{lineage}_LC.csv"
df.to_csv(output_filename, index=False)

# PGT121

## Import and Calculate data

In [24]:
lineage = 'PGT121'
# -------------------------------------------------------
# Import RMSD data for the mature antibody. (Mature antibody first because the PDB we are comparing to is the mature antibody)
system_name=f'{lineage}_mature_kappa_VC'
HC_start, HC_end, LC_start, LC_end = 1, 132, 1, 105 # Mature 
mature_VHC_VLC_resi_names, mature_HC_rmsd_values, mature_LC_rmsd_values = combined_steps_to_import_data(system_name, HC_start, HC_end, LC_start, LC_end)

# Calculate minimum distances based on the published PDB structure:
print('Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.')
pdb_path = f'{BASE_PATH}structures/7uoj_{lineage}_clean_renumbered.pdb'
HC, LC, antigen = '4', '3', '0'
pdb_HC_glycan_minDists, pdb_LC_glycan_minDists, pdb_HC_resis, pdb_LC_resis = calculate_min_dist_values(pdb_path, HC, LC, antigen, HC_end, LC_end)
# -------------------------------------------------------
# Align the PDB data to the mature antibody -- some PDB structures are missing the first few residues or last few resis. 
check_pdb_lengths_mature_Abs(mature_VHC_VLC_resi_names, pdb_HC_resis, pdb_LC_resis, n=3)
# HC PDB is already aligned.
# Need to add 1 and remove 1 resi for the LC.
pdb_LC_resis_fixed           = np.insert(pdb_LC_resis,           0,'GLN1')
pdb_LC_glycan_minDists_fixed = np.insert(pdb_LC_glycan_minDists, 0, None)
for i in range(LC_end):
    if str(pdb_LC_resis_fixed[i]) != str(mature_VHC_VLC_resi_names[1][i]):
        print('ERROR: LC misalignment',i, pdb_LC_resis_fixed[i], mature_VHC_VLC_resi_names[1][i])
# -------------------------------------------------------
# Import RMSD data for the naive antibody now.
system_name=f'{lineage}_naive_kappa_VC'
HC_start, HC_end, LC_start, LC_end = 1, 132, 1, 109 # Naive 
naive_VHC_VLC_resi_names, naive_HC_rmsd_values, naive_LC_rmsd_values = combined_steps_to_import_data(system_name, HC_start, HC_end, LC_start, LC_end)

Importing RMSD data for PGT121_mature_kappa_VC
	INPUTS: apo/PGT121_mature_kappa_VC 1 132 1 105
Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.
133 104 331
		Variable Heavy Chain:	Variable Light Chain:
Mature Length:	 132 			 105
   PDB Length:	 132 			 104
 Mature Start:	 [GLN1, MET2, GLN3] 		 [SER1, ASP2, ILE3]
    PDB Start:	 [GLN1, MET2, GLN3] 		 [ASP2, ILE3, SER4]
   Mature End:	 [VAL130, SER131, SER132] 	 [THR103, VAL104, LEU105]
      PDB End:	 [VAL130, SER131, SER132] 	 [THR103, VAL104, LEU105] 

ERROR: LC misalignment 0 GLN1 SER1
Importing RMSD data for PGT121_naive_kappa_VC
	INPUTS: apo/PGT121_naive_kappa_VC 1 132 1 109


## Align Naive and Mature/PDB (sequence-based alignment)

In [25]:
# Heavy chain for PGT121 is already aligned. Just save the output. 
# Light chain for PGT121 needs to be aligned. Adding a total of 4 'None' values to mature sequence. Also to PDB mature sequence. 
mature_LC_rmsd_values_fixed=np.insert(mature_LC_rmsd_values,    [0,0,0,0 ,0,0,0,0, 90,90], None)
mature_VLC_resi_names_fixed=np.insert(mature_VHC_VLC_resi_names[1], [0,0,0,0 ,0,0,0,0, 90,90], None)

naive_LC_rmsd_values_fixed = np.insert(naive_LC_rmsd_values, [9, 64, 66,66,97,97], None)
naive_VLC_resi_names_fixed = np.insert(naive_VHC_VLC_resi_names[1], [9, 64, 66,66,97,97], None)

pdb_LC_resis_fixed_fixed          =np.insert(pdb_LC_resis_fixed,          [0,0,0,0 ,0,0,0,0, 90,90], None)
pdb_LC_glycan_minDists_fixed_fixed=np.insert(pdb_LC_glycan_minDists_fixed,[0,0,0,0 ,0,0,0,0, 90,90], None)

## Save output for plotting later. 

In [26]:
data={"pdb_mature_HC_resi":pdb_HC_resis,
      "pdb_mature_HC_glycan_minDists":pdb_HC_glycan_minDists,
      "naive_HC_resi": naive_VHC_VLC_resi_names[0],
      "naive_HC_rmsd": naive_HC_rmsd_values,
      "mature_HC_resi": mature_VHC_VLC_resi_names[0],
      "mature_HC_rmsd": mature_HC_rmsd_values}
df = pd.DataFrame(data)
output_filename = f"{BASE_PATH}data/to_glycan/{lineage}_HC.csv"
df.to_csv(output_filename, index=False)

data={"pdb_mature_LC_resi":pdb_LC_resis_fixed_fixed,
      "pdb_mature_LC_glycan_minDists":pdb_LC_glycan_minDists_fixed_fixed,
      "naive_LC_resi": naive_VLC_resi_names_fixed,
      "naive_LC_rmsd": naive_LC_rmsd_values_fixed,
      "mature_LC_resi": mature_VLC_resi_names_fixed,
      "mature_LC_rmsd": mature_LC_rmsd_values_fixed}
df = pd.DataFrame(data)
output_filename = f"{BASE_PATH}data/to_glycan/{lineage}_LC.csv"
df.to_csv(output_filename, index=False)

# PGT124

In [27]:
lineage = 'PGT124'
# -------------------------------------------------------
# Import RMSD data for the mature antibody. (Mature antibody first because the PDB we are comparing to is the mature antibody)
system_name=f'{lineage}_mature_VC'
HC_start, HC_end, LC_start, LC_end = 1, 132, 1, 108 # Mature 
mature_VHC_VLC_resi_names, mature_HC_rmsd_values, mature_LC_rmsd_values = combined_steps_to_import_data(system_name, HC_start, HC_end, LC_start, LC_end)

# Calculate minimum distances based on the published PDB structure:
print('Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.')
pdb_path = f'{BASE_PATH}structures/8d50_PGT124_clean_vc_renumbered_N332.pdb' 
HC, LC, antigen = '2', '1', '0'
pdb_HC_glycan_minDists, pdb_LC_glycan_minDists, pdb_HC_resis, pdb_LC_resis = calculate_min_dist_values(pdb_path, HC, LC, antigen, HC_end, LC_end)
# -------------------------------------------------------

Importing RMSD data for PGT124_mature_VC
	INPUTS: apo/PGT124_mature_VC 1 132 1 108
Calculating heavy atom minimum distances based on the published PDB structure and saving a list of the residues in the antibody.
131 107 61


In [28]:
# -------------------------------------------------------
# Align the PDB data to the mature antibody -- some PDB structures are missing the first few residues or last few resis. 
check_pdb_lengths_mature_Abs(mature_VHC_VLC_resi_names, pdb_HC_resis, pdb_LC_resis, n=4)
# HC PDB is already aligned.
# Need to add 1 for the HC and LC.
pdb_LC_resis_fixed           = np.insert(pdb_LC_resis,           0,'SER1')
pdb_LC_glycan_minDists_fixed = np.insert(pdb_LC_glycan_minDists, 0, None)
for i in range(LC_end):
    if str(pdb_LC_resis_fixed[i]) != str(mature_VHC_VLC_resi_names[1][i]):
        print('ERROR: LC misalignment',i, pdb_LC_resis_fixed[i], mature_VHC_VLC_resi_names[1][i])

# Need to add 1 for the HC and LC.
pdb_HC_resis_fixed           = np.insert(pdb_HC_resis,           0,'GLN1')
pdb_HC_glycan_minDists_fixed = np.insert(pdb_HC_glycan_minDists, 0, None)
for i in range(HC_end):
    if str(pdb_HC_resis_fixed[i]) != str(mature_VHC_VLC_resi_names[0][i]):
        print('ERROR: LC misalignment',i, pdb_HC_resis_fixed[i], mature_VHC_VLC_resi_names[0][i])


		Variable Heavy Chain:	Variable Light Chain:
Mature Length:	 132 			 108
   PDB Length:	 131 			 107
 Mature Start:	 [GLN1, VAL2, GLN3, LEU4] 		 [SER1, TYR2, VAL3, SER4]
    PDB Start:	 [VAL2, GLN3, LEU4, GLN5] 		 [TYR2, VAL3, SER4, PRO5]
   Mature End:	 [THR129, VAL130, SER131, SER132] 	 [LEU105, THR106, VAL107, LEU108]
      PDB End:	 [THR129, VAL130, SER131, SER132] 	 [LEU105, THR106, VAL107, LEU108] 



In [29]:
# -------------------------------------------------------
# Import RMSD data for the naive antibody now.
system_name=f'PGT121_naive_kappa_VC'
HC_start, HC_end, LC_start, LC_end = 1, 132, 1, 109 # Naive 
naive_VHC_VLC_resi_names, naive_HC_rmsd_values, naive_LC_rmsd_values = combined_steps_to_import_data(system_name, HC_start, HC_end, LC_start, LC_end)

Importing RMSD data for PGT121_naive_kappa_VC
	INPUTS: apo/PGT121_naive_kappa_VC 1 132 1 109


In [30]:
mature_LC_rmsd_values_fixed=np.insert(mature_LC_rmsd_values,    [3,3,3,3], None)
mature_VLC_resi_names_fixed=np.insert(mature_VHC_VLC_resi_names[1], [3,3,3,3], None)

In [31]:
pdb_LC_resis_fixed_fixed          =np.insert(pdb_LC_resis_fixed,          [3,3,3,3], None)
pdb_LC_glycan_minDists_fixed_fixed=np.insert(pdb_LC_glycan_minDists_fixed,[3,3,3,3], None)

In [32]:
print(len(pdb_HC_glycan_minDists_fixed), len(mature_VHC_VLC_resi_names[0]))

132 132


In [33]:
data={"pdb_mature_LC_resi":pdb_LC_resis_fixed_fixed,
      "pdb_mature_LC_glycan_minDists":pdb_LC_glycan_minDists_fixed_fixed,
      "mature_LC_resi": mature_VLC_resi_names_fixed,
      "mature_LC_rmsd": mature_LC_rmsd_values_fixed}
df = pd.DataFrame(data)
output_filename = f"{BASE_PATH}data/to_glycan/PGT124_LC.csv"
df.to_csv(output_filename, index=False)

In [34]:
data={"pdb_mature_HC_resi":pdb_HC_resis_fixed,
      "pdb_mature_HC_glycan_minDists":pdb_HC_glycan_minDists_fixed,
      "mature_HC_resi": mature_VHC_VLC_resi_names[0],
      "mature_HC_rmsd": mature_HC_rmsd_values}
df = pd.DataFrame(data)
output_filename = f"{BASE_PATH}data/to_glycan/PGT124_HC.csv"
df.to_csv(output_filename, index=False)

In [35]:
print(len(pdb_LC_glycan_minDists_fixed_fixed))

112


In [36]:
for i in range(len(pdb_HC_glycan_minDists_fixed)):
    if pdb_HC_glycan_minDists_fixed[i] < 6:
        print(i, pdb_HC_resis_fixed[i])

102 ARG103
103 ILE104
104 TYR105
105 GLY106
106 VAL107
107 VAL108
111 GLU112


In [37]:
LC: 
SER25
ARG26
ASN45
ASN46
GLN47

THR60
PRO61
ASP62
ILE63
ASN64

PHE65



HC:
ARG103
ILE104
TYR105
GLY106
VAL107
VAL108
GLU112

103-108+112

SyntaxError: invalid syntax (3563567682.py, line 1)